In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("auto24_ebay_predicted.csv")

In [4]:
from sklearn.model_selection import train_test_split

numeric_features = ['power_ps', 'mileage_km', 'car_age', "ebay_logprice_predicted"]
categorical_features = ['body_type', 'transmission', 'model', 'fuel_type', 'make', 'damaged', ]
#data[categorical_features] = data[categorical_features].fillna("unknown")

y = data['price_eur'].copy()
X = data[categorical_features + numeric_features].copy()

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, random_state = 42, test_size = 0.2)

In [5]:
#Standard HistGradientBoostRegressor
import time
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import ParameterGrid

ohe = OrdinalEncoder(handle_unknown="use_encoded_value",
    unknown_value=np.nan,
    encoded_missing_value=np.nan,
    max_categories=254,)
cat_indices = list(range(len(numeric_features), len(numeric_features) + len(categorical_features)))

seed = 42

def make_forest(loss = "absolute_error", learning_rate = 0.1, max_iter= 300, max_leaf_nodes = 31, min_samples_leaf = 5, l2_regularization=0.0,):
    prepare = ColumnTransformer([
        ("numeric", "passthrough", numeric_features),
        ("categorical", ohe, categorical_features),
    ])
    pipe = Pipeline([
        ("prepare", prepare),
        ("forest", HistGradientBoostingRegressor(loss=loss,
            learning_rate=learning_rate,
            max_iter=max_iter,
            max_leaf_nodes=max_leaf_nodes,
            min_samples_leaf=min_samples_leaf,
            l2_regularization=l2_regularization,
            categorical_features=cat_indices,
            early_stopping=True,      
            validation_fraction=0.1,
            n_iter_no_change=15,
            random_state=seed,)),
         ])
    return pipe
        
#Defining the hyperparameters and running the model:

def holdout_table(parameter_choices):
    rows, models = [], []
    for i, settings in enumerate(ParameterGrid(parameter_choices), start=1):
        t0 = time.time()
        model = make_forest(**settings).fit(X_train, y_train)
        models.append(model)
        probs_train = model.predict(X_train)
        probs_val = model.predict(X_val)
        
        predictors = model.named_steps["forest"]._predictors
        
        rows.append({
            "settings":settings,
            "training MAE": mean_absolute_error(y_train, probs_train), #which measure to use?!
            "validation MAE": mean_absolute_error(y_val, probs_val), #which measure to use?!
            "validation MAE in %": mean_absolute_percentage_error(y_val, probs_val)*100,
            "training RMSE": root_mean_squared_error(y_train, probs_train),
            "validation RMSE": root_mean_squared_error(y_val, probs_val),
            "val_r2": r2_score(y_val, probs_val),
            "avg_nodes": sum(p[0].nodes.shape[0] for p in predictors) / len(predictors),
        })
        print(f"[{i}/9] Finished {settings} in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows), models

depth_results, depth_models = holdout_table({
    "loss" : ["absolute_error"], "learning_rate": [0.1], "max_iter" : [1000], "max_leaf_nodes" : [255], "min_samples_leaf" : [20,], "l2_regularization": [0.1],
})
leaf_labels = [str(settings["max_leaf_nodes"]) for settings in depth_results["settings"]]

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(depth_results.round(3))




[1/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.1, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 255, 'min_samples_leaf': 20} in 18.8s


,settings,training MAE,validation MAE,validation MAE in %,training RMSE,validation RMSE,val_r2,avg_nodes
0,"{'l2_regularization': 0.1, 'learning_rate': 0....",1586.218,3159.544,19.481,3728.69,5968.444,0.892,507.383
